# Data Audit & Vector Extraction Analysis

This notebook verifies data quality and analyzes the two extraction methods:
- **matched_lines**: Mean activations from response start to last edited line
- **k_after_edit**: Mean activations over K tokens after last edited line

**Key validations**:
1. Data statistics match expected counts
2. Both vectors properly window on-policy and off-policy
3. No systematic length/truncation confounds
4. Layer-wise performance comparison
5. Edit count stratification analysis

## 1. Setup

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('../data')
ARTIFACT_DIR = Path('../artifacts')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

## 2. Load and Validate Data

In [2]:
# Load all data files
with open(DATA_DIR / 'on_policy.json') as f:
    on_policy_data = json.load(f)

with open(DATA_DIR / 'off_policy.json') as f:
    off_policy_data = json.load(f)

with open(DATA_DIR / 'sentence_paraphrases.json') as f:
    paraphrase_db = json.load(f)

# Load vectors
with open(ARTIFACT_DIR / 'vector_matched_lines.json') as f:
    vec_matched = json.load(f)

with open(ARTIFACT_DIR / 'vector_k_after_edit.json') as f:
    vec_k_after = json.load(f)

print("=" * 60)
print("DATA STATISTICS")
print("=" * 60)
print(f"Prompts: {len(on_policy_data)}")
print(f"On-policy rollouts: {sum(len(ex['rollouts']) for ex in on_policy_data)}")
print(f"Off-policy prompts: {len(off_policy_data)}")
print(f"Off-policy on-policy rollouts: {sum(len(ex['on_policy']) for ex in off_policy_data)}")
print(f"Off-policy variants: {sum(len(ex['off_policy']) for ex in off_policy_data)}")
print(f"Unique sentences paraphrased: {len(paraphrase_db)}")
print(f"Total paraphrases: {sum(len(v) for v in paraphrase_db.values())}")
print(f"\nSentences with all 3 paraphrases: {sum(1 for v in paraphrase_db.values() if len(v) == 3)} / {len(paraphrase_db)}")
print("\n" + "=" * 60)
print("VECTOR INFORMATION")
print("=" * 60)
print(f"Matched Lines Vector:")
print(f"  Layer: {vec_matched['layer']}")
print(f"  Window: {vec_matched['window']}")
print(f"  Dimension: {len(vec_matched['vector'])}")
print(f"  Layers considered: {vec_matched['layers_considered'][0]}-{vec_matched['layers_considered'][-1]}")
print(f"\nK-After-Edit Vector:")
print(f"  Layer: {vec_k_after['layer']}")
print(f"  Window: {vec_k_after['window']}")
print(f"  K tokens: {vec_k_after['k_tokens']}")
print(f"  Dimension: {len(vec_k_after['vector'])}")
print(f"  Layers considered: {vec_k_after['layers_considered'][0]}-{vec_k_after['layers_considered'][-1]}")

DATA STATISTICS
Prompts: 40
On-policy rollouts: 200
Off-policy prompts: 40
Off-policy on-policy rollouts: 200
Off-policy variants: 600
Unique sentences paraphrased: 6821
Total paraphrases: 20463

Sentences with all 3 paraphrases: 6821 / 6821

VECTOR INFORMATION
Matched Lines Vector:
  Layer: 35
  Window: matched_lines
  Dimension: 2560
  Layers considered: 18-35

K-After-Edit Vector:
  Layer: 35
  Window: k_after_edit
  K tokens: 24
  Dimension: 2560
  Layers considered: 18-35
